# ECG Neuromorphic System

**Workflow:** Run Cells 1–9 once (install, define, load, train). Then re-run Cells 10–13 freely to test different thresholds without retraining.

### Cell 1: Install Dependencies (run once)

In [ ]:
!pip install wfdb numpy torch snntorch

### Cell 2: Imports & Constants

In [ ]:
import wfdb
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import snntorch as snn
from snntorch import surrogate, functional as SF

DS1 = ['101','106','108','109','112','114','115','116','118','119','122','124',
       '201','203','205','207','208','209','215','220','223','230']

DS2 = ['100','103','105','111','113','117','121','123','200','202','210','212',
       '213','214','219','221','222','228','231','232','233','234']

AAMI = {'N':0,'L':0,'R':0,'e':0,'j':0,
        'A':1,'a':1,'J':1,'S':1,
        'V':2,'E':2,
        'F':3,
        '/':4,'f':4,'Q':4}

### Cell 3: Helper Functions

In [ ]:
def load_records(records, win=200):
    X, y = [], []
    half = win // 2
    for r in records:
        print(f"Loading record {r}")
        rec = wfdb.rdrecord(r, pn_dir='mitdb')
        ann = wfdb.rdann(r, 'atr', pn_dir='mitdb')
        sig = rec.p_signal[:, 0]
        for s, sym in zip(ann.sample, ann.symbol):
            if sym in AAMI and s-half > 0 and s+half < len(sig):
                X.append(sig[s-half:s+half])
                y.append(AAMI[sym])
    return np.array(X), np.array(y)

def quantize(X):
    Xmin = X.min(axis=1, keepdims=True)
    Xmax = X.max(axis=1, keepdims=True)
    Xn = (X - Xmin) / (Xmax - Xmin + 1e-6)
    return (Xn * 2047).astype(np.int32)

class ECGDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

### Cell 4: Delta Modulator (Spike Encoder)

In [ ]:
class DeltaMod:
    def __init__(self, step=4):
        self.step = step

    def encode(self, x):
        x = x.long()
        B, T = x.shape
        spikes = torch.zeros(T, B, 2)
        signal = torch.zeros(B, dtype=torch.long)
        init = torch.zeros(B, dtype=torch.bool)
        last = torch.zeros(B, dtype=torch.bool)

        for t in range(T):
            ecg = x[:, t]
            up = torch.zeros(B)
            dn = torch.zeros(B)

            init_mask = ~init
            signal[init_mask] = ecg[init_mask]
            init[init_mask] = True

            active = init & (~last)
            cu = active & (ecg > signal + self.step)
            cd = active & (ecg + self.step < signal)

            up[cu] = 1
            dn[cd] = 1

            signal = torch.where(cu, torch.clamp(signal + self.step, max=2047), signal)
            signal = torch.where(cd, torch.clamp(signal - self.step, min=0), signal)

            last = cu | cd
            spikes[t,:,0] = up
            spikes[t,:,1] = dn

        return spikes

### Cell 5: Training Model (SNN with Surrogate Gradients)

In [ ]:
class DPE_Train(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2, 5, bias=False)
        self.lif = snn.Leaky(beta=0.95,
                             spike_grad=surrogate.fast_sigmoid(),
                             init_hidden=True)

    def forward(self, spikes):
        self.lif.reset_hidden()
        out = []
        for t in range(spikes.size(0)):
            cur = self.fc(spikes[t])
            spk = self.lif(cur)
            out.append(spk)
        return torch.stack(out)

### Cell 6: Hardware Blocks (Synapse, Neuron, WTA)

In [ ]:
class Synapse:
    def __init__(self, w, learn=3):
        self.w = w
        self.syn_time = 7
        self.LEARN = learn
        self.lut = [1,2,3,0,-2,-1,0]

    def step(self, spike, neuron_time, refractory, stdp=False):
        if spike:
            self.syn_time = 0
        elif self.syn_time < 7:
            self.syn_time += 1

        if stdp:
            dt = neuron_time - self.syn_time
            idx = self.LEARN if abs(dt) > self.LEARN else dt + self.LEARN
            cond = ((spike) or neuron_time == 0) and (abs(dt) <= self.LEARN)
            if cond:
                self.w += self.lut[idx]
                if self.w == 0:
                    self.w = 1

        return self.w if spike else 0


class Neuron:
    def __init__(self, thresh=20, leak_rate=2, refractory=0):
        self.acc = 0
        self.thresh = thresh
        self.leak_rate = leak_rate
        self.ref = refractory
        self.time = 10
        self.leak_counter = 0
        self.fire = 0

    def step(self, s0, s1, any_fire):
        pre = s0 + s1
        if self.fire:
            self.fire = 0
        else:
            self.fire = int(self.acc >= self.thresh)

        if self.fire or any_fire:
            self.acc = 0
            self.leak_counter = 0
            self.time = 0
        elif self.time > self.ref:
            nxt = self.acc + pre
            if self.leak_counter < self.leak_rate:
                self.acc = nxt
                self.leak_counter += 1
            else:
                self.leak_counter = 0
                if nxt > 0:
                    self.acc = nxt - 1
                elif nxt < 0:
                    self.acc = nxt + 1
                else:
                    self.acc = nxt
            self.time += 1
        else:
            self.time += 1

        return self.fire


def WTA(fires):
    winner = [0]*len(fires)
    any_fire = any(fires)
    if any_fire:
        for i, f in enumerate(fires):
            if f:
                winner[i] = 1
                break
    return winner, any_fire

### Cell 7: Hardware Network

In [ ]:
class HW_Net:
    def __init__(self, weights, stdp=False):
        self.stdp = stdp
        self.syn = [[Synapse(weights[i][j]) for j in range(2)] for i in range(5)]
        self.neurons = [Neuron() for _ in range(5)]

    def step(self, spike):
        syn_out = []
        for i in range(5):
            s0 = self.syn[i][0].step(spike[0], self.neurons[i].time, self.neurons[i].ref, self.stdp)
            s1 = self.syn[i][1].step(spike[1], self.neurons[i].time, self.neurons[i].ref, self.stdp)
            syn_out.append((s0, s1))

        fires = []
        for i in range(5):
            fires.append(self.neurons[i].step(syn_out[i][0], syn_out[i][1], False))

        winner, any_fire = WTA(fires)
        if any_fire:
            for i in range(5):
                if not winner[i]:
                    self.neurons[i].acc = 0

        return winner

---
## Load Data & Train (run once, then skip)

### Cell 8: Load & Prepare Data

In [ ]:
print("Loading data...")
Xtr, ytr = load_records(DS1)
Xte, yte = load_records(DS2)

Xtr = quantize(Xtr)
Xte = quantize(Xte)

train_loader = DataLoader(ECGDataset(Xtr, ytr), batch_size=32, shuffle=True)
test_loader  = DataLoader(ECGDataset(Xte, yte), batch_size=32)

encoder = DeltaMod()
print("Data ready.")

### Cell 9: Train (weighted loss, 30 epochs, lr=5e-4)

In [ ]:
# Compute class weights (inverse frequency)
class_counts = Counter(ytr.tolist())
total_samples = len(ytr)
num_classes = 5

class_weights = torch.zeros(num_classes)
for c in range(num_classes):
    if class_counts[c] > 0:
        class_weights[c] = total_samples / (num_classes * class_counts[c])

print("Class weights:", class_weights)

model = DPE_Train()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
loss_fn = SF.ce_count_loss(weight=class_weights)

print("Training...")
for epoch in range(30):
    total_loss = 0
    for Xb, yb in train_loader:
        Xb = Xb.clone().detach()
        yb = yb.clone().detach()
        spikes = encoder.encode(Xb)
        optimizer.zero_grad()
        out = model(spikes)
        loss = loss_fn(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}: Loss={total_loss:.3f}")

print("Training complete.")

---
## Hardware Evaluation (re-run freely below this line)

### Cell 10: Scale Weights (6-bit) & Inspect

In [ ]:
# --- TUNABLE PARAMETERS ---
WEIGHT_BITS = 6          # 6-bit signed: range [-32, 31]
MAX_POS = 2**(WEIGHT_BITS-1) - 1  # 31
MAX_NEG = -(2**(WEIGHT_BITS-1))   # -32

raw_weights = model.fc.weight.detach().numpy()
max_abs = np.max(np.abs(raw_weights))
SCALE = MAX_POS / max_abs
scaled_weights = np.clip(np.round(raw_weights * SCALE).astype(int), MAX_NEG, MAX_POS)

print("Raw weights:")
print(raw_weights)
print(f"\nScale factor: {SCALE:.4f}")
print(f"\nScaled weights ({WEIGHT_BITS}-bit signed, range [{MAX_NEG}, {MAX_POS}]):")
print(scaled_weights)

### Cell 11: Threshold Sweep

In [ ]:
print("Sweeping thresholds...\n")

for th in [5, 10, 15, 20, 30, 50, 75, 100, 150, 200]:
    net = HW_Net(scaled_weights, False)
    for n in net.neurons:
        n.thresh = th
    correct = 0
    total = 0
    for Xb, yb in test_loader:
        Xb = Xb.clone().detach()
        spikes = encoder.encode(Xb)
        for i in range(Xb.shape[0]):
            counts = np.zeros(5)
            for t in range(spikes.shape[0]):
                w = net.step(spikes[t, i])
                counts += w
            pred = np.argmax(counts)
            if pred == yb[i]:
                correct += 1
            total += 1
    print(f"thresh={th:3d}  Accuracy={correct/total:.4f}")

### Cell 12: Evaluate Best Threshold (with and without STDP)

In [ ]:
# Set this to the best threshold from the sweep above
BEST_THRESH = 20

def evaluate(thresh, stdp=False):
    net = HW_Net(scaled_weights, stdp)
    for n in net.neurons:
        n.thresh = thresh
    correct = 0
    total = 0
    for Xb, yb in test_loader:
        Xb = Xb.clone().detach()
        spikes = encoder.encode(Xb)
        for i in range(Xb.shape[0]):
            counts = np.zeros(5)
            for t in range(spikes.shape[0]):
                w = net.step(spikes[t, i])
                counts += w
            pred = np.argmax(counts)
            if pred == yb[i]:
                correct += 1
            total += 1
    return correct / total

print(f"HW Accuracy (no STDP):   {evaluate(BEST_THRESH, False):.4f}")
print(f"HW Accuracy (with STDP): {evaluate(BEST_THRESH, True):.4f}")

### Cell 13: Confusion Matrix & Per-Class Accuracy

In [ ]:
# Uses BEST_THRESH from Cell 12
print("Test set class distribution:")
print(Counter(yte.tolist()))

net = HW_Net(scaled_weights, False)
for n in net.neurons:
    n.thresh = BEST_THRESH

all_preds = []
all_labels = []

for Xb, yb in test_loader:
    Xb = Xb.clone().detach()
    spikes = encoder.encode(Xb)
    for i in range(Xb.shape[0]):
        counts = np.zeros(5)
        for t in range(spikes.shape[0]):
            w = net.step(spikes[t, i])
            counts += w
        all_preds.append(np.argmax(counts))
        all_labels.append(yb[i].item())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Confusion matrix
class_names = ['N', 'S', 'V', 'F', 'Q']
num_classes = 5
cm = np.zeros((num_classes, num_classes), dtype=int)
for true, pred in zip(all_labels, all_preds):
    cm[true][pred] += 1

print("\nConfusion Matrix (rows=true, cols=predicted):")
print(f"{'':>8}", end="")
for name in class_names:
    print(f"{name:>8}", end="")
print()
for i, name in enumerate(class_names):
    print(f"{name:>8}", end="")
    for j in range(num_classes):
        print(f"{cm[i][j]:>8}", end="")
    print()

# Per-class accuracy
print("\nPer-class accuracy:")
for i, name in enumerate(class_names):
    total_cls = cm[i].sum()
    correct_cls = cm[i][i]
    acc = correct_cls / total_cls if total_cls > 0 else 0
    print(f"  {name}: {correct_cls}/{total_cls} = {acc:.4f}")

print(f"\nOverall: {np.trace(cm)}/{cm.sum()} = {np.trace(cm)/cm.sum():.4f}")